[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/10_gqa.ipynb)

# 🔴 Hard: Grouped Query Attention (GQA)

Implement **Grouped Query Attention** — used in LLaMA 2, Mistral, etc. to reduce KV cache size.

Like MHA, but with **fewer KV heads** than Q heads. Each group of Q heads shares the same K/V head.

### Signature
```python
class GroupQueryAttention:
    def __init__(self, d_model: int, num_heads: int, num_kv_heads: int): ...
    def forward(self, x) -> torch.Tensor:  # self-attention
```

### Requirements
- `self.W_q`: `nn.Linear(d_model, d_model)` — full Q projection
- `self.W_k`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced K projection
- `self.W_v`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced V projection
- `self.W_o`: `nn.Linear(d_model, d_model)` — output projection
- `d_k = d_model // num_heads`
- Expand KV heads with `repeat_interleave` to match Q heads
- When `num_kv_heads == num_heads`, should behave like standard MHA

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [1]:
import torch
import torch.nn as nn
import math

e:\Graduate\MAC同步\文稿\1.code\TorchCode\.venv\Lib\site-packages\torch\_subclasses\functional_tensor.py:368: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [4]:
# ✏️ YOUR IMPLEMENTATION HERE

class GroupQueryAttention:
    def __init__(self, d_model: int, num_heads: int, num_kv_heads: int):
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model,d_model)
        self.W_k = nn.Linear(d_model,self.d_k*num_kv_heads)
        self.W_v = nn.Linear(d_model,self.d_k*num_kv_heads)
        
        self.W_o = nn.Linear(d_model,d_model)
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.group_size = num_heads // num_kv_heads

    def forward(self, x) -> torch.Tensor:
        # (batchsize,seq_len,d_model) 
        q:torch.Tensor = self.W_q(x)
        # (batchsize,seq_len,d_k*num_kv_heads) 
        k:torch.Tensor = self.W_k(x)
        v:torch.Tensor = self.W_v(x)
        
        batch_size,se_q,d_model = q.shape
        se_k = k.shape[1]
        
        # (batchsize,head_num,seq_len,d_k) 
        q = q.reshape(batch_size,se_q,self.num_heads,-1).transpose(1,2)
        # (batchsize,num_kv_heads,seq_len,d_k) 
        k = k.reshape(batch_size,se_k,self.num_kv_heads,self.d_k).transpose(1,2)
        v = v.reshape(batch_size,se_k,self.num_kv_heads,self.d_k).transpose(1,2)
        
        k = k.repeat_interleave(repeats=self.group_size,dim=1)
        v = v.repeat_interleave(repeats=self.group_size,dim=1)
         # (batchsize,head_num,seq_len,seq_len) 
        score = torch.softmax(q@k.transpose(-2,-1)/math.sqrt(self.d_k),dim=-1)
        # (batchsize,head_num,seq_len,d_k) 
        content = score @ v
        
        content = content.transpose(1,2).contiguous().view(batch_size,se_q,-1)
        return self.W_o(content)

In [5]:
# 🧪 Debug
torch.manual_seed(0)
gqa = GroupQueryAttention(d_model=32, num_heads=8, num_kv_heads=2)
print("W_q shape:", gqa.W_q.weight.shape)  # (32, 32)
print("W_k shape:", gqa.W_k.weight.shape)  # (8, 32)  — only 2 KV heads * d_k=4

x = torch.randn(2, 6, 32)
out = gqa.forward(x)
print("Output shape:", out.shape)           # (2, 6, 32)

W_q shape: torch.Size([32, 32])
W_k shape: torch.Size([8, 32])
Output shape: torch.Size([2, 6, 32])


In [6]:
from torch_judge import check
check('gqa')


🧪 Testing: Grouped Query Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (3.5ms)
  ✅ [2/5] nn.Linear with correct shapes (0.6ms)
  ✅ [3/5] Degenerates to MHA when kv_heads == heads (5.3ms)
  ✅ [4/5] KV heads are shared correctly (5.6ms)
  ✅ [5/5] Gradient flow (3.8ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (18.7ms total)
  Progress saved. Run status() to see your dashboard.

